In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from scipy import stats

In [2]:
df_d = pd.read_csv("Dutch_paydiff_norm.csv")
df_h = pd.read_csv("Honolulu_paydiff_norm.csv")
df_d = df_d[df_d["subsession.round_number"] > 2]
df_h = df_h[df_h["subsession.round_number"] > 2]

### bid and price deviation regression

In [41]:
df_d["is_high_cost"] = (df_d["session.config.discount_b"] == 0.019)
df_d["item_value_high_cost"] = df_d["is_high_cost"] * df_d["player.item_value"]
df_d["is_5_bidder"] = df_d["session.config.name"].str.contains("5_")
df_d["mid_paydiff"] = df_d.apply(
        lambda x: np.median(
            df_d[(df_d["is_high_cost"] == x["is_high_cost"]) & (df_d["is_5_bidder"] == x["is_5_bidder"])]["paydiff_pct"].unique()
        ), 
        axis=1,
    )
df_d["is_top"] = (df_d["paydiff_pct"] > df_d["mid_paydiff"])
df_d["is_dutch_first"] = df_d["session.config.name"].str.contains("_D1")

In [42]:
df_h["is_high_cost"] = (df_h["session.config.discount_b"] == 0.019)
df_h["item_value_high_cost"] = df_h["is_high_cost"] * df_h["player.item_value"]
df_h["is_5_bidder"] = df_h["session.config.name"].str.contains("5_")
df_h["mid_paydiff"] = df_h.apply(
        lambda x: np.median(
            df_h[(df_h["is_high_cost"] == x["is_high_cost"]) & (df_h["is_5_bidder"] == x["is_5_bidder"])]["paydiff_pct"].unique()
        ), 
        axis=1,
    )
df_h["is_top"] = (df_h["paydiff_pct"] > df_h["mid_paydiff"])
df_h["item_value_top"] = df_h["is_top"] * df_h["player.item_value"]
df_h["is_dutch_first"] = df_h["session.config.name"].str.contains("_D1")

In [76]:
def regDBid(df, is_5_bidder, is_top):

    tmp = df[(df["player.is_dutch_winner"] == 1) & (df["is_5_bidder"] == is_5_bidder) & (df["is_top"] == is_top)]
    
    cols = [
        "player.item_value", "is_high_cost", "item_value_high_cost", "is_dutch_first",
    ]
    y = tmp["group.dutch_final_price"].astype(float)
    x = tmp[cols].astype(float)

    x = sm.add_constant(x) # constant is added to the first column !!!
    model = sm.OLS(y, x, missing="raise")

    return model

In [ ]:
regDBid(df_d, is_5_bidder=1, is_top=0).fit(
    use_t=True,
).summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               OLS Regression Results                              
===================================================================================
Dep. Variable:     group.dutch_final_price   R-squared:                       0.449
Model:                                 OLS   Adj. R-squared:                  0.437
Method:                      Least Squares   F-statistic:                     37.84
Date:                       周三, 30 4月 2025   Prob (F-statistic):           3.81e-23
Time:                             15:58:54   Log-Likelihood:                -599.68
No. Observations:                      191   AIC:                             1209.
Df Residuals:                          186   BIC:                             1226.
Df Model:                                4                                         
Covariance Type:                 nonrobust                                         
========================================================================================
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   30.9646      2.209     14.014      0.000      26.606      35.323
player.item_value        0.2064      0.053      3.894      0.000       0.102       0.311
is_high_cost           -24.6934      3.628     -6.806      0.000     -31.851     -17.536
item_value_high_cost     0.5428      0.088      6.163      0.000       0.369       0.717
is_dutch_first           1.0968      0.924      1.187      0.237      -0.725       2.919
==============================================================================
Omnibus:                       17.223   Durbin-Watson:                   1.579
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               59.683
Skew:                          -0.084   Prob(JB):                     1.10e-13
Kurtosis:                       5.733   Cond. No.                         444.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
def regDstageBid(df, is_5_bidder, is_high_cost):

    tmp = df[(df["player.is_dutch_winner"] == 1) & (df["is_high_cost"] == is_high_cost) & (df["is_5_bidder"] == is_5_bidder)]

    cols = [
        "player.item_value", "is_top", "item_value_top", "is_dutch_first",
    ]
    y = tmp["group.dutch_final_price"].astype(float)
    x = tmp[cols].astype(float)

    x = sm.add_constant(x) # constant is added to the first column !!!
    model = sm.OLS(y, x, missing="raise")

    return model

In [ ]:
regDstageBid(df_h, is_5_bidder=1, is_high_cost=1).fit(
    use_t=True,
).summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               OLS Regression Results                              
===================================================================================
Dep. Variable:     group.dutch_final_price   R-squared:                       0.335
Model:                                 OLS   Adj. R-squared:                  0.323
Method:                      Least Squares   F-statistic:                     25.88
Date:                     Wed, 30 Apr 2025   Prob (F-statistic):           2.26e-17
Time:                             15:45:06   Log-Likelihood:                -658.96
No. Observations:                      210   AIC:                             1328.
Df Residuals:                          205   BIC:                             1345.
Df Model:                                4                                         
Covariance Type:                 nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                10.4412      2.088      4.999      0.000       6.323      14.559
player.item_value     0.3989      0.057      6.947      0.000       0.286       0.512
is_top                2.5800      3.000      0.860      0.391      -3.335       8.495
item_value_top       -0.0386      0.082     -0.471      0.638      -0.200       0.123
is_dutch_first       -2.3182      0.784     -2.958      0.003      -3.863      -0.773
==============================================================================
Omnibus:                       20.884   Durbin-Watson:                   1.194
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               25.392
Skew:                          -0.705   Prob(JB):                     3.06e-06
Kurtosis:                       3.957   Cond. No.                         361.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [100]:
def regEstageDrop(df, is_5_bidder, is_top, is_pooled_topbot):

    tmp = df[
        (df["group.have_contest_winner"] > 1) & (df["player.contest_status"] == 1) 
        & ((df["player.is_english_winner"] == 0) | (df_h["player.dropout_price.accurate"] == 50))
    ]
    if is_pooled_topbot:
        tmp = tmp[(tmp["is_5_bidder"] == is_5_bidder)]
    else:
        tmp = tmp[(tmp["is_5_bidder"] == is_5_bidder) & (tmp["is_top"] == is_top)]
    
    cols = [
        "player.item_value", "is_high_cost", "item_value_high_cost", "is_dutch_first",
    ]
    y = tmp["player.dropout_price"].astype(float)
    x = tmp[cols].astype(float)

    x = sm.add_constant(x) # constant is added to the first column !!!
    model = sm.OLS(y, x, missing="raise")

    return model

In [ ]:
regEstageDrop(df_h, is_5_bidder=1, is_top=1, is_pooled_topbot=0).fit(
    use_t=True,
).summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                             
================================================================================
Dep. Variable:     player.dropout_price   R-squared:                       0.745
Model:                              OLS   Adj. R-squared:                  0.741
Method:                   Least Squares   F-statistic:                     221.8
Date:                    周三, 30 4月 2025   Prob (F-statistic):           7.97e-89
Time:                          19:14:55   Log-Likelihood:                -838.64
No. Observations:                   309   AIC:                             1687.
Df Residuals:                       304   BIC:                             1706.
Df Model:                             4                                         
Covariance Type:              nonrobust                                         
========================================================================================
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    8.2250      1.241      6.627      0.000       5.783      10.667
player.item_value        0.7777      0.036     21.509      0.000       0.707       0.849
is_high_cost             2.4321      1.653      1.471      0.142      -0.821       5.685
item_value_high_cost    -0.0837      0.050     -1.669      0.096      -0.182       0.015
is_dutch_first          -1.1729      0.438     -2.676      0.008      -2.035      -0.310
==============================================================================
Omnibus:                      215.815   Durbin-Watson:                   2.078
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             2571.790
Skew:                           2.767   Prob(JB):                         0.00
Kurtosis:                      16.004   Cond. No.                         341.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Mann-Whitney U test

In [117]:
def rankSumTestH(rmin, rmax, n, altHD, altHF):
    tmpdf = df_h[
        (df_h["subsession.round_number"] >= rmin) 
        & (df_h["subsession.round_number"] <= rmax) 
        & (df_h["session.config.name"].str.contains(str(n) + "_"))]
    tmpdf_HD_D1 = tmpdf[(tmpdf["player.is_dutch_winner"] == 1) & (tmpdf["session.config.name"].str.contains("D1H2"))][[
        "session.code", "subsession.round_number", "group.id_in_subsession", 
        "group.dutch_final_price", "predict.player.optimal_dutch_bid"
    ]]
    tmpdf_HD_D2 = tmpdf[(tmpdf["player.is_dutch_winner"] == 1) & (tmpdf["session.config.name"].str.contains("D2H1"))][[
        "session.code", "subsession.round_number", "group.id_in_subsession", 
        "group.dutch_final_price", "predict.player.optimal_dutch_bid"
    ]]
    tmpdf_HF_D1 = tmpdf[tmpdf["session.config.name"].str.contains("D1H2")][[
        "session.code", "subsession.round_number", "group.id_in_subsession", 
        "group.final_price", "predict.group.final_price"
    ]].drop_duplicates()
    tmpdf_HF_D2 = tmpdf[tmpdf["session.config.name"].str.contains("D2H1")][[
        "session.code", "subsession.round_number", "group.id_in_subsession", 
        "group.final_price", "predict.group.final_price"
    ]].drop_duplicates()

    tmplist_D1, tmplist_D2 = [], []
    for code in set(tmpdf_HD_D1["session.code"]):
        actual_bid = np.mean(tmpdf_HD_D1[tmpdf_HD_D1["session.code"] == code]["group.dutch_final_price"])
        predict_bid = np.mean(tmpdf_HD_D1[tmpdf_HD_D1["session.code"] == code]["predict.player.optimal_dutch_bid"])
        actual_final = np.mean(tmpdf_HF_D1[tmpdf_HF_D1["session.code"] == code]["group.final_price"])
        predict_final = np.mean(tmpdf_HF_D1[tmpdf_HF_D1["session.code"] == code]["predict.group.final_price"])
        tmplist_D1.append([
            code,
            actual_bid - predict_bid,
            actual_final - predict_final
        ])
    for code in set(tmpdf_HD_D2["session.code"]):
        actual_bid = np.mean(tmpdf_HD_D2[tmpdf_HD_D2["session.code"] == code]["group.dutch_final_price"])
        predict_bid = np.mean(tmpdf_HD_D2[tmpdf_HD_D2["session.code"] == code]["predict.player.optimal_dutch_bid"])
        actual_final = np.mean(tmpdf_HF_D2[tmpdf_HF_D2["session.code"] == code]["group.final_price"])
        predict_final = np.mean(tmpdf_HF_D2[tmpdf_HF_D2["session.code"] == code]["predict.group.final_price"])
        tmplist_D2.append([
            code,
            actual_bid - predict_bid,
            actual_final - predict_final
        ])
    testdf_D1 = pd.DataFrame(tmplist_D1, columns=["treatment", "bidDiff", "finalDiff"])
    testdf_D2 = pd.DataFrame(tmplist_D2, columns=["treatment", "bidDiff", "finalDiff"])

    print(testdf_D1)
    print(testdf_D2)

    print("%s-bidder H D-stage bid test" % n)
    print(stats.mannwhitneyu(testdf_D1["bidDiff"], testdf_D2["bidDiff"], alternative=altHD))
    print("%s-bidder H final price test" % n)
    print(stats.mannwhitneyu(testdf_D1["finalDiff"], testdf_D2["finalDiff"], alternative=altHF))

In [118]:
rankSumTestH(3, 30, 2, "two-sided", "two-sided")
rankSumTestH(3, 30, 5, "two-sided", "two-sided")

  treatment   bidDiff  finalDiff
0  7xbozwc7 -3.535245  -2.224544
1  c9oliyzc -3.182615  -3.916144
2  yq06059v -1.278910  -2.470245
3  nli0vb0t -4.872931  -3.592221
4  5m969a1u -0.851309  -1.560599
  treatment   bidDiff  finalDiff
0  785y9gdo -1.206228  -2.266624
1  p66lw61j -3.363375  -3.571420
2  cbdt6t3s  0.577299  -1.565078
2-bidder H D-stage bid test
MannwhitneyuResult(statistic=4.0, pvalue=0.39285714285714285)
2-bidder H final price test
MannwhitneyuResult(statistic=6.0, pvalue=0.7857142857142857)
  treatment   bidDiff  finalDiff
0  w4lqxa01  1.326830   0.267473
1  a2ehf0h4 -1.419408  -1.986968
2  15t0rnnr -3.772173  -1.398889
3  hie4719f  0.614259  -1.434459
  treatment   bidDiff  finalDiff
0  btnhfeao  0.197439   0.013485
1  ogo32l47  1.970272  -1.043356
2  r6ovli5x  0.950872  -0.295081
5-bidder H D-stage bid test
MannwhitneyuResult(statistic=3.0, pvalue=0.4)
5-bidder H final price test
MannwhitneyuResult(statistic=3.0, pvalue=0.4)


In [52]:
def rankSumTestD(rmin, rmax, n, altHD, altHF):
    tmpdf = df_d[
        (df_d["subsession.round_number"] >= rmin) 
        & (df_d["subsession.round_number"] <= rmax) 
        & (df_d["session.config.name"].str.contains(str(n) + "_"))]
    tmpdf_HD_D1 = tmpdf[(tmpdf["player.is_dutch_winner"] == 1) & (tmpdf["session.config.name"].str.contains("D1H2"))][[
        "session.code", "subsession.round_number", "group.id_in_subsession", 
        "group.dutch_final_price", "predict.player.bid"
    ]]
    tmpdf_HD_D2 = tmpdf[(tmpdf["player.is_dutch_winner"] == 1) & (tmpdf["session.config.name"].str.contains("D2H1"))][[
        "session.code", "subsession.round_number", "group.id_in_subsession", 
        "group.dutch_final_price", "predict.player.bid"
    ]]
    tmpdf_HF_D1 = tmpdf[tmpdf["session.config.name"].str.contains("D1H2")][[
        "session.code", "subsession.round_number", "group.id_in_subsession", 
        "group.dutch_final_price", "predict.group.dutch_final_price"
    ]].drop_duplicates()
    tmpdf_HF_D2 = tmpdf[tmpdf["session.config.name"].str.contains("D2H1")][[
        "session.code", "subsession.round_number", "group.id_in_subsession", 
        "group.dutch_final_price", "predict.group.dutch_final_price"
    ]].drop_duplicates()

    tmplist_D1, tmplist_D2 = [], []
    for code in set(tmpdf_HD_D1["session.code"]):
        actual_bid = np.mean(tmpdf_HD_D1[tmpdf_HD_D1["session.code"] == code]["group.dutch_final_price"])
        predict_bid = np.mean(tmpdf_HD_D1[tmpdf_HD_D1["session.code"] == code]["predict.player.bid"])
        actual_final = np.mean(tmpdf_HF_D1[tmpdf_HF_D1["session.code"] == code]["group.dutch_final_price"])
        predict_final = np.mean(tmpdf_HF_D1[tmpdf_HF_D1["session.code"] == code]["predict.group.dutch_final_price"])
        tmplist_D1.append([
            code,
            actual_bid - predict_bid,
            actual_final - predict_final
        ])
    for code in set(tmpdf_HD_D2["session.code"]):
        actual_bid = np.mean(tmpdf_HD_D2[tmpdf_HD_D2["session.code"] == code]["group.dutch_final_price"])
        predict_bid = np.mean(tmpdf_HD_D2[tmpdf_HD_D2["session.code"] == code]["predict.player.bid"])
        actual_final = np.mean(tmpdf_HF_D2[tmpdf_HF_D2["session.code"] == code]["group.dutch_final_price"])
        predict_final = np.mean(tmpdf_HF_D2[tmpdf_HF_D2["session.code"] == code]["predict.group.dutch_final_price"])
        tmplist_D2.append([
            code,
            actual_bid - predict_bid,
            actual_final - predict_final
        ])
    testdf_D1 = pd.DataFrame(tmplist_D1, columns=["treatment", "bidDiff", "finalDiff"])
    testdf_D2 = pd.DataFrame(tmplist_D2, columns=["treatment", "bidDiff", "finalDiff"])

    print("%s-bidder D bid test" % n)
    print(stats.mannwhitneyu(testdf_D1["bidDiff"], testdf_D2["bidDiff"], alternative=altHD))
    print("%s-bidder D final price test" % n)
    print(stats.mannwhitneyu(testdf_D1["finalDiff"], testdf_D2["finalDiff"], alternative=altHF))

In [53]:
rankSumTestD(3, 30, 2, "two-sided", "two-sided")
rankSumTestD(3, 30, 5, "two-sided", "two-sided")

2-bidder D bid test
MannwhitneyuResult(statistic=11.0, pvalue=0.39285714285714285)
2-bidder D final price test
MannwhitneyuResult(statistic=12.0, pvalue=0.25)
5-bidder D bid test
MannwhitneyuResult(statistic=6.0, pvalue=1.0)
5-bidder D final price test
MannwhitneyuResult(statistic=5.0, pvalue=0.8571428571428571)


### bid and price deviation bootstrap

In [3]:
def genData(df_d, df_h):

    df_d["is_high_cost"] = (df_d["session.config.discount_b"] == 0.019)
    df_d["item_value_high_cost"] = df_d["player.item_value"] * df_d["is_high_cost"]
    df_d["is_5_bidder"] = df_d["session.config.name"].str.contains("5_")
    df_d["is_dfirst"] = df_d["session.config.name"].str.contains("D1H2")
    df_d["mid_paydiff"] = df_d.apply(
        lambda x: np.median(
            df_d[(df_d["is_high_cost"] == x["is_high_cost"]) & (df_d["is_5_bidder"] == x["is_5_bidder"])]["paydiff_pct"].unique()
        ), 
        axis=1,
    )
    df_d["is_top"] = (df_d["paydiff_pct"] > df_d["mid_paydiff"])
    df_d["item_value_top"] = df_d["player.item_value"] * df_d["is_top"]

    df_h["is_high_cost"] = (df_h["session.config.discount_b"] == 0.019)
    df_h["item_value_high_cost"] = df_h["player.item_value"] * df_h["is_high_cost"]
    df_h["is_5_bidder"] = df_h["session.config.name"].str.contains("5_")
    df_h["is_dfirst"] = df_h["session.config.name"].str.contains("D1H2")
    df_h["mid_paydiff"] = df_h.apply(
        lambda x: np.median(
            df_h[(df_h["is_high_cost"] == x["is_high_cost"]) & (df_h["is_5_bidder"] == x["is_5_bidder"])]["paydiff_pct"].unique()
        ), 
        axis=1,
    )
    df_h["is_top"] = (df_h["paydiff_pct"] > df_h["mid_paydiff"])
    df_h["item_value_top"] = df_h["player.item_value"] * df_h["is_top"]

    # for dutch auction bids
    cols = [
    "group.dutch_final_price", "predict.player.bid",
    "player.item_value", "is_high_cost", "is_5_bidder", "is_dfirst", "is_top", 
    "item_value_high_cost", "item_value_top",
    "session.code",
    ]
    droplist = [] # ['4uwxoyym', 'twmqn1o6', '3yrhqskd', 'wgrvj2qu']
    df_dbid = df_d[(df_d["player.is_dutch_winner"] == 1) & (~df_d["participant.code"].isin(droplist))][cols]

    # for honolulu dutch stage bids
    cols = [
    "group.dutch_final_price", "predict.player.optimal_dutch_bid",
    "player.item_value", "is_high_cost", "is_5_bidder", "is_dfirst", "is_top", 
    "item_value_high_cost", "item_value_top",
    "session.code",
    ]
    droplist = [] # ['ece1g596', '4uwxoyym', 'co853jdf', 'ovsmhled']
    df_hbid = df_h[(df_h["player.is_dutch_winner"] == 1) & (~df_h["participant.code"].isin(droplist))][cols]

    # for dutch auction final prices
    cols = [
    "group.dutch_final_price", "predict.group.dutch_final_price",
    "is_high_cost", "is_5_bidder", "is_dfirst",
    "session.code", "subsession.round_number", "group.id_in_subsession",
    ]
    droplist = [] # ['4uwxoyym', 'twmqn1o6', '3yrhqskd', 'wgrvj2qu']
    df_dprice = df_d[(~df_d["participant.code"].isin(droplist))][cols].drop_duplicates()

    # for honolulu final prices
    cols = [
    "group.final_price", "predict.group.final_price",
    "is_high_cost", "is_5_bidder", "is_dfirst",
    "session.code", "subsession.round_number", "group.id_in_subsession",
    ]
    droplist = [] # ['ece1g596', '4uwxoyym', 'co853jdf', 'ovsmhled']
    df_hprice = df_h[(~df_h["participant.code"].isin(droplist))][cols].drop_duplicates()

    return df_dbid, df_hbid, df_dprice, df_hprice

In [4]:
df_dbid, df_hbid, df_dprice, df_hprice = genData(df_d, df_h)

In [5]:
def regDbid(df):
    cols = [
        #"player.item_value", "is_high_cost", "item_value_high_cost",
        #"player.item_value", "is_top", "item_value_top",
        "is_dfirst",
    ]
    y = (df["group.dutch_final_price"] - df["predict.player.bid"]).astype(float)
    x = df[cols].astype(float)

    x = sm.add_constant(x) # constant is added to the first column !!!
    model = sm.OLS(y, x, missing="raise")

    return model

def regHbid(df):
    cols = [
        #"player.item_value", "is_high_cost", "item_value_high_cost",
        #"player.item_value", "is_top", "item_value_top",
        "is_dfirst",
    ]
    y = (df["group.dutch_final_price"] - df["predict.player.optimal_dutch_bid"]).astype(float)
    x = df[cols].astype(float)

    x = sm.add_constant(x) # constant is added to the first column !!!
    model = sm.OLS(y, x, missing="raise")

    return model

def regDprice(df):
    cols = [
        #"is_high_cost",
        "is_dfirst",
    ]
    y = (df["group.dutch_final_price"] - df["predict.group.dutch_final_price"]).astype(float)
    x = df[cols].astype(float)

    x = sm.add_constant(x) # constant is added to the first column !!!
    model = sm.OLS(y, x, missing="raise")

    return model

def regHprice(df):
    cols = [
        #"is_high_cost",
        "is_dfirst",
    ]
    y = (df["group.final_price"] - df["predict.group.final_price"]).astype(float)
    x = df[cols].astype(float)

    x = sm.add_constant(x) # constant is added to the first column !!!
    model = sm.OLS(y, x, missing="raise")

    return model

In [6]:
def sampleReg(df, type):

    # resampling data on the session level (the block) for each treatment
    sessions = df["session.code"].unique()
    flag = False
    while not flag:
        sessions_sample = pd.DataFrame({"session.code" : np.random.choice(sessions, size=sessions.size, replace=True)})
        df_sample = sessions_sample.merge(df, how="left", on="session.code")
        # set the IF conditions depending on what baseline regression is used to avoid having variables with single value 
        if (len(set(df_sample["is_dfirst"])) == 2):
        # if (len(set(df_sample["is_dfirst"])) == 2) and (len(set(df_sample["is_high_cost"])) == 2):
            flag = True

    global TMPSAMPLE
    TMPSAMPLE = df_sample

    model = eval("reg%s(df_sample)" % type)

    return model

In [7]:
def baselineReg(df, is_5_bidder, is_high_cost, is_pooled_highlow, type):
    
    if is_pooled_highlow:
        tmp = df[(df["is_5_bidder"] == is_5_bidder)]
    else:
        tmp = df[(df["is_5_bidder"] == is_5_bidder) & (df["is_high_cost"] == is_high_cost)]

    model = eval("reg%s(tmp)" % type)

    return model

def baselineReg2(df, is_5_bidder, is_top, is_pooled_topbot, type):
    
    if is_pooled_topbot:
        tmp = df[(df["is_5_bidder"] == is_5_bidder)]
    else:
        tmp = df[(df["is_5_bidder"] == is_5_bidder) & (df["is_top"] == is_top)]

    model = eval("reg%s(tmp)" % type)

    return model

In [8]:
def bootstrap(df, is_5_bidder, is_high_cost, is_pooled_highlow, type, rep):
    
    if is_pooled_highlow:
        tmp = df[(df["is_5_bidder"] == is_5_bidder)]
    else:
        tmp = df[(df["is_5_bidder"] == is_5_bidder) & (df["is_high_cost"] == is_high_cost)]
    
    cols = ["const", "is_dfirst"]
    bs_predictions = pd.DataFrame(columns=cols)
    for i in range(rep):
        # constant is in the first column !!!
        model = sampleReg(tmp, type)
        bs_predictions.loc[len(bs_predictions)] = [model.fit().params["const"], model.fit().params["is_dfirst"]]
        
    return bs_predictions

def bootstrap2(df, is_5_bidder, is_top, is_pooled_topbot, type, rep):
    
    if is_pooled_topbot:
        tmp = df[(df["is_5_bidder"] == is_5_bidder)]
    else:
        tmp = df[(df["is_5_bidder"] == is_5_bidder) & (df["is_top"] == is_top)]
    
    cols = ["const", "is_dfirst"]
    bs_predictions = pd.DataFrame(columns=cols)
    for i in range(rep):
        # constant is in the first column !!!
        model = sampleReg(tmp, type)
        bs_predictions.loc[len(bs_predictions)] = [model.fit().params["const"], model.fit().params["is_dfirst"]]
        
    return bs_predictions

In [13]:
def bootstrapT(df, is_5_bidder, is_high_cost, is_pooled_highlow, type, rep):

    # baseline test statistic -- theta_base
    model_base = baselineReg(df, is_5_bidder, is_high_cost, is_pooled_highlow, type)
    df_t = model_base.fit().df_resid

    # bootstrap test statistics -- theta_bs
    bs_predictions = bootstrap(df, is_5_bidder, is_high_cost, is_pooled_highlow, type, rep)

    # bootstrap standard error, p-value
    theta_base = model_base.fit().params["is_dfirst"]
    theta_bs = bs_predictions["is_dfirst"]
    se_bs = np.std(theta_bs, ddof=1)
    print([theta_base, se_bs, stats.t.sf(abs(theta_base) / se_bs, df=df_t) * 2])

def bootstrapT2(df, is_5_bidder, is_top, is_pooled_topbot, type, rep):

    # baseline test statistic -- theta_base
    model_base = baselineReg2(df, is_5_bidder, is_top, is_pooled_topbot, type)
    df_t = model_base.fit().df_resid

    # bootstrap test statistics -- theta_bs
    bs_predictions = bootstrap2(df, is_5_bidder, is_top, is_pooled_topbot, type, rep)

    # bootstrap standard error, p-value
    theta_base = model_base.fit().params["is_dfirst"]
    theta_bs = bs_predictions["is_dfirst"]
    se_bs = np.std(theta_bs, ddof=1)
    print([theta_base, se_bs, stats.t.sf(abs(theta_base) / se_bs, df=df_t) * 2])

D bids, 2-bidder

In [11]:
np.random.seed(1)
print("pooled")
bootstrapT(df_dbid, is_5_bidder=0, is_high_cost=1, is_pooled_highlow=1, type="Dbid", rep=1000)
print("high")
bootstrapT(df_dbid, is_5_bidder=0, is_high_cost=1, is_pooled_highlow=0, type="Dbid", rep=1000)
print("low")
bootstrapT(df_dbid, is_5_bidder=0, is_high_cost=0, is_pooled_highlow=0, type="Dbid", rep=1000)

pooled
[1.3120923037512782, 1.829763529911918, 0.4735655546119474]
high
[0.6453259951963597, 2.0146496210518126, 0.7489182949177005]
low
[0.5497783589720046, 0.11770986323425107, 4.348994021259084e-06]


In [14]:
np.random.seed(1)
print("pooled")
bootstrapT2(df_dbid, is_5_bidder=0, is_top=1, is_pooled_topbot=1, type="Dbid", rep=1000)
print("top")
bootstrapT2(df_dbid, is_5_bidder=0, is_top=1, is_pooled_topbot=0, type="Dbid", rep=1000)
print("bot")
bootstrapT2(df_dbid, is_5_bidder=0, is_top=0, is_pooled_topbot=0, type="Dbid", rep=1000)

pooled
[1.3120923037512782, 1.829763529911918, 0.4735655546119474]
top
[0.004108759636363368, 1.7569762516402894, 0.9981354338136151]
bot
[2.7579083816277974, 2.349195282168611, 0.24123942996149927]


D bids, 5-bidder

In [149]:
np.random.seed(1)
print("pooled")
bootstrapT(df_dbid, is_5_bidder=1, is_high_cost=1, is_pooled_highlow=1, type="Dbid", rep=1000)
print("high")
bootstrapT(df_dbid, is_5_bidder=1, is_high_cost=1, is_pooled_highlow=0, type="Dbid", rep=1000)
print("low")
bootstrapT(df_dbid, is_5_bidder=1, is_high_cost=0, is_pooled_highlow=0, type="Dbid", rep=1000)

pooled
[1.2902902596399066, 1.6372756680001395, 0.43111704077710467]
high
[-0.9538307221280834, 0.34987918401917806, 0.006953910614401592]
low
[3.3330846337261915, 2.026170809517855, 0.10158569955053348]


In [15]:
np.random.seed(1)
print("pooled")
bootstrapT2(df_dbid, is_5_bidder=1, is_top=1, is_pooled_topbot=1, type="Dbid", rep=1000)
print("top")
bootstrapT2(df_dbid, is_5_bidder=1, is_top=1, is_pooled_topbot=0, type="Dbid", rep=1000)
print("bot")
bootstrapT2(df_dbid, is_5_bidder=1, is_top=0, is_pooled_topbot=0, type="Dbid", rep=1000)

pooled
[1.2902902596399066, 1.6372756680001395, 0.43111704077710467]
top
[-0.18921734779365953, 0.5893971030894787, 0.7484983197031972]
bot
[2.594475232441142, 2.298263824907413, 0.260377246625937]


H bids, 2-bidder

In [146]:
np.random.seed(1)
print("pooled")
bootstrapT(df_hbid, is_5_bidder=0, is_high_cost=1, is_pooled_highlow=1, type="Hbid", rep=1000)
print("high")
bootstrapT(df_hbid, is_5_bidder=0, is_high_cost=1, is_pooled_highlow=0, type="Hbid", rep=1000)
print("low")
bootstrapT(df_hbid, is_5_bidder=0, is_high_cost=0, is_pooled_highlow=0, type="Hbid", rep=1000)

pooled
[-1.4031641602993377, 1.1937416448441382, 0.24025495169436747]
high
[-2.68164625897717, 1.6438334763942297, 0.10376742385534739]
low
[-0.5116256332841608, 0.6338088531385315, 0.420154438516505]


In [17]:
np.random.seed(1)
print("pooled")
bootstrapT2(df_hbid, is_5_bidder=0, is_top=1, is_pooled_topbot=1, type="Hbid", rep=1000)
print("top")
bootstrapT2(df_hbid, is_5_bidder=0, is_top=1, is_pooled_topbot=0, type="Hbid", rep=1000)
print("bot")
bootstrapT2(df_hbid, is_5_bidder=0, is_top=0, is_pooled_topbot=0, type="Hbid", rep=1000)

pooled
[-1.4031641602993377, 1.1937416448441382, 0.24025495169436747]
top
[-1.7207805514553103, 1.0375636742983603, 0.09818203830539093]
bot
[-0.9441427878466565, 1.9924318759943598, 0.6359227161384947]


H bids, 5-bidder

In [147]:
np.random.seed(1)
print("pooled")
bootstrapT(df_hbid, is_5_bidder=1, is_high_cost=1, is_pooled_highlow=1, type="Hbid", rep=1000)
print("high")
bootstrapT(df_hbid, is_5_bidder=1, is_high_cost=1, is_pooled_highlow=0, type="Hbid", rep=1000)
print("low")
bootstrapT(df_hbid, is_5_bidder=1, is_high_cost=0, is_pooled_highlow=0, type="Hbid", rep=1000)

pooled
[-1.7744646474756949, 1.1353747020562217, 0.11886165290762674]
high
[-2.196308947145498, 2.064613365113888, 0.2886586002316283]
low
[-1.353445826035713, 0.8138101271311988, 0.09790752432270787]


In [18]:
np.random.seed(1)
print("pooled")
bootstrapT2(df_hbid, is_5_bidder=1, is_top=1, is_pooled_topbot=1, type="Hbid", rep=1000)
print("top")
bootstrapT2(df_hbid, is_5_bidder=1, is_top=1, is_pooled_topbot=0, type="Hbid", rep=1000)
print("bot")
bootstrapT2(df_hbid, is_5_bidder=1, is_top=0, is_pooled_topbot=0, type="Hbid", rep=1000)

pooled
[-1.7744646474756949, 1.1353747020562217, 0.11886165290762674]
top
[0.2689273504224138, 0.8599922934358701, 0.7548376307634838]
bot
[-3.6599295995272314, 2.364844549138113, 0.12322897171592495]


D price, 2-bidder

In [150]:
np.random.seed(1)
print("pooled")
bootstrapT(df_dprice, is_5_bidder=0, is_high_cost=1, is_pooled_highlow=1, type="Dprice", rep=1000)
print("high")
bootstrapT(df_dprice, is_5_bidder=0, is_high_cost=1, is_pooled_highlow=0, type="Dprice", rep=1000)
print("low")
bootstrapT(df_dprice, is_5_bidder=0, is_high_cost=0, is_pooled_highlow=0, type="Dprice", rep=1000)

pooled
[1.6263723885963155, 1.792464445308562, 0.36453994531102274]
high
[1.2206556943944435, 1.7543587940283931, 0.487015209319721]
low
[0.4760136524116845, 0.12833962056523682, 0.00024292265870815157]


D price, 5-bidder

In [151]:
np.random.seed(1)
print("pooled")
bootstrapT(df_dprice, is_5_bidder=1, is_high_cost=1, is_pooled_highlow=1, type="Dprice", rep=1000)
print("high")
bootstrapT(df_dprice, is_5_bidder=1, is_high_cost=1, is_pooled_highlow=0, type="Dprice", rep=1000)
print("low")
bootstrapT(df_dprice, is_5_bidder=1, is_high_cost=0, is_pooled_highlow=0, type="Dprice", rep=1000)

pooled
[0.3754024898121215, 1.1922865234044953, 0.7530310441206359]
high
[-1.27871795752177, 0.17631316512798487, 7.932728393502144e-12]
low
[1.902900658809523, 1.5835094218828474, 0.23094532151417002]


H price, 2-bidder

In [152]:
np.random.seed(1)
print("pooled")
bootstrapT(df_hprice, is_5_bidder=0, is_high_cost=1, is_pooled_highlow=1, type="Hprice", rep=1000)
print("high")
bootstrapT(df_hprice, is_5_bidder=0, is_high_cost=1, is_pooled_highlow=0, type="Hprice", rep=1000)
print("low")
bootstrapT(df_hprice, is_5_bidder=0, is_high_cost=0, is_pooled_highlow=0, type="Hprice", rep=1000)

pooled
[-0.298861841360506, 0.6388701662282654, 0.6400750403664317]
high
[-0.3401338428500004, 0.9579580415808528, 0.7227527407280165]
low
[-0.36862255015313583, 0.5997890130874645, 0.5392385374640838]


H-price, 5-bidder

In [153]:
np.random.seed(1)
print("pooled")
bootstrapT(df_hprice, is_5_bidder=1, is_high_cost=1, is_pooled_highlow=1, type="Hprice", rep=1000)
print("high")
bootstrapT(df_hprice, is_5_bidder=1, is_high_cost=1, is_pooled_highlow=0, type="Hprice", rep=1000)
print("low")
bootstrapT(df_hprice, is_5_bidder=1, is_high_cost=0, is_pooled_highlow=0, type="Hprice", rep=1000)

pooled
[-0.7030901364415366, 0.5441273521548926, 0.19704638191030513]
high
[0.016712943924165433, 0.7704459121884856, 0.9827139942836349]
low
[-1.415632154940476, 0.2210965790997875, 1.1238271378561244e-09]
